# Coleta de Dados e Pandas

In [ ]:
# Apenas para MacOS, para resolver o problema de certificado SSL
# import os
# os.environ['SSL_CERT_FILE'] = '/etc/ssl/cert.pem'
# os.environ['REQUESTS_CA_BUNDLE'] = '/etc/ssl/cert.pem'

In [ ]:
# biblioteca para requisição
# !pip install --upgrade pip
# !pip install requests
# !pip install matplotlib
# !pip install openpyxl
# !pip install --upgrade certifi
# !pip install missingno
# !pip install scikit-learn

## Importações de Bibliotecas

In [ ]:
import requests
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns
sns.set_theme(style="whitegrid")

import missingno as msno
from scipy import stats
from sklearn.impute import KNNImputer

# 1. Carregar Base de Dados

In [ ]:
url = "https://raw.githubusercontent.com/prof-mrafaelbatista/261_UNIESP_MBA-DADOS_data_mining_inferencia_estatistica/refs/heads/main/datasets/OnlineRetail.csv"

In [ ]:
df = pd.read_csv(url, encoding='latin1')

## Valores Ausentes

In [ ]:
# 1. Detecção de missings
print(df.isnull().sum())
print((df.isnull().mean()*100).round(2))  # percentual

In [ ]:
# 2. Visualização com missingno
msno.matrix(df)
plt.show()

In [ ]:
# 3. Estratégias de imputação
# Imputar mediana em UnitPrice (poucos missings)
df['UnitPrice'] = df['UnitPrice'].fillna(df['UnitPrice'].median())

In [ ]:
# Remover linhas sem CustomerID (muitos missings, MNAR)
df_limpo = df.dropna(subset=['CustomerID'])
print(f"Linhas removidas: {len(df)-len(df_limpo)}")

In [ ]:
# KNNImputer (exemplo com colunas numéricas)
imputer = KNNImputer(n_neighbors=5)
df_num = df[['Quantity','UnitPrice']].copy()
df_num_imp = pd.DataFrame(imputer.fit_transform(df_num), columns=df_num.columns)

In [ ]:
# 4. Outliers com Z-Score
z_scores = np.abs(stats.zscore(df_limpo['Quantity'].dropna()))
outliers_z = (z_scores > 3).sum()
print(f"Outliers por Z-Score (|z|>3): {outliers_z}")

In [ ]:
# Filtrar
df_sem_outliers = df_limpo[np.abs(stats.zscore(df_limpo['Quantity'])) <= 3]
print(f"Shape após remoção outliers: {df_sem_outliers.shape}")

In [ ]:
# 5. Boxplot antes e depois
fig, axes = plt.subplots(1, 2, figsize=(12,4))
df_limpo[df_limpo['Quantity']>0]['Quantity'].plot(kind='box', ax=axes[0], title='Antes')
df_sem_outliers[df_sem_outliers['Quantity']>0]['Quantity'].plot(kind='box', ax=axes[1], title='Depois')
plt.show()